# 16 · Spark joins in depth (Spark)

How Spark runs a join:
- **Broadcast hash join**: the small side is copied to every executor. No shuffle. Fastest.
- **Sort-merge join**: both sides are **shuffled** (moved over the network) by the key, sorted, then merged.
  The default for two big tables.
- **Skew**: one key has far more rows than the others, so one task does most of the work and the job waits for it.

`how=` values: `inner`, `left`, `right`, `full`, `left_semi`, `left_anti`, `cross`.

**How to run:** Start a cell with `%%sql` to write Spark SQL. In a normal Python cell use the DataFrame API: the tables are ready as variables (`employees`, `orders`, ...). Put your answer in `res` and call `res.show()`.

Tables: see [tables_guide.md](../../cheatsheets/tables_guide.md). Run the setup cell first (Shift+Enter).

In [ ]:
import sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
            if (p / "settings.py").exists())   # the repo folder
sys.path.insert(0, str(ROOT / "spark"))
from spark_setup import get_spark, load_tables
from pyspark.sql import functions as F, Window, types as T

spark = get_spark()
t = load_tables(spark)          # also makes SQL views with the same names
employees, departments = t["employees"], t["departments"]
customers, orders = t["customers"], t["orders"]
products, order_items = t["products"], t["order_items"]
user_logins = t["user_logins"]
%load_ext spark_setup

## Q1. All join types in one table  ·  🟢 easy

Join `customers` to the **distinct customer_ids** of `orders`. Print the row count for each `how`: inner, left, right, full, left_semi, left_anti.

In [ ]:
# your answer here (DataFrame API)

## Q2. Ambiguous column names  ·  🟡 medium

Join employees to departments with an **expression** (`e.dept_id == d.dept_id`) and select `dept_id`. It fails: why? Fix it two ways: (1) join on the column **name**, (2) use aliases.

In [ ]:
# your answer here (DataFrame API)

## Q3. NULL keys never match  ·  🟡 medium

Make two small DataFrames whose key column has a NULL. Join them with `==` and with `eqNullSafe`. Compare the results.

*Expected result: 2 rows · columns: `va`, `vb`*

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q4. Join on two columns with different names  ·  🟢 easy

Join `order_items` (order_id, product_id) to a small price list `prices(oid, pid, unit_price)` on both keys. Show matching rows.

*Expected result: 2 rows · columns: `order_id`, `product_id`, `quantity`, `unit_price`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q5. Broadcast or not? Read the plan  ·  🟡 medium

Join `order_items` to `products` (1) with AQE broadcast **turned off** (`spark.sql.autoBroadcastJoinThreshold = -1`) and (2) with `F.broadcast`. Print the physical plan of both and find `SortMergeJoin` vs `BroadcastHashJoin`.

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q6. Find skewed keys  ·  🟡 medium

Make a fake big table: 1,000,000 rows where key `0` has 90% of the rows. Count rows per key and show the top 3 keys.

In [ ]:
# your answer here (DataFrame API)

## Q7. Fix skew with salting  ·  🔴 hard

Join the skewed `big` table to a small `dim(key, label)` with **salting**: add a random salt 0–9 to `big`, copy each `dim` row 10 times (one per salt), join on (key, salt). Check the result count is still 1,000,000.

In [ ]:
# your answer here (DataFrame API)

## Q8. Range join (non-equi)  ·  🟡 medium

Give each employee a salary band from a band table: `low` (0–79,999), `mid` (80,000–129,999), `high` (130,000+). Join on `salary BETWEEN min AND max`.

*Expected result: 30 rows · columns: `name`, `salary`, `band`*

In [ ]:
%%sql
-- your answer here

In [ ]:
# DataFrame API: put your answer in res
res = ...
res.show()

## Q9. Cross join explosion  ·  🟢 easy

How many rows do `customers × products` make? And `orders × order_items` (without a condition)? Print both counts.

In [ ]:
# your answer here (DataFrame API)